# Линейные модели с нуля: сравнение с sklearn

Ноутбук проверяет собственные `LinearRegression` и `LogisticRegression`
(пакет `linear_models`) на реальных и синтетических данных. Каждая модель
обучается и проверяется на **тех же данных**, что и её аналог из `sklearn`.

Разделы:
1. Данные
2. Линейная регрессия: `analytic` и `gradient` против `LinearRegression` и `Ridge`
3. Логистическая регрессия против `sklearn.linear_model.LogisticRegression`
4. Режимы: без штрафа, L1, L2, с `intercept` и без
5. Сходимость: `loss_history_` и `learning_rate`
6. Выводы и объяснение расхождений

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import LinearRegression as SkLinearRegression
from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.linear_model import Ridge as SkRidge
from sklearn.linear_model import Lasso as SkLasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from linear_models.linear_regression import LinearRegression
from linear_models.logistic_regression import LogisticRegression

SEED = 42
np.set_printoptions(precision=6, suppress=True)

print("python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)
print("свои классы :", LinearRegression.__name__, LogisticRegression.__name__)
print("sklearn     :", SkLinearRegression.__name__, SkRidge.__name__, SkLogisticRegression.__name__)

## 1. Данные

Четыре набора: для регрессии синтетика с известными весами и `load_diabetes`,
для классификации синтетика и `load_breast_cancer`.

- Одно разбиение train/test на набор: `test_size=0.25`, `random_state=SEED`
  (для классификации со `stratify=y`, чтобы доли классов совпадали).
- Признаки стандартизируются: **среднее и стандартное отклонение считаются
  только по train**, к test применяется то же преобразование.
- Наши модели и модели `sklearn` получают **одни и те же** массивы.
- Масштабирование нужно из-за градиентного спуска с фиксированным
  `learning_rate`: на сырых признаках `load_breast_cancer` он не успевает
  сойтись (это показано в разделе про сходимость).

In [ ]:
TEST_SIZE = 0.25


def split_and_scale(X, y, stratify=None):
    """Одно разбиение train/test и стандартизация по train."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=stratify
    )
    scaler = StandardScaler().fit(X_train)
    return {
        "X_train": scaler.transform(X_train),
        "X_test": scaler.transform(X_test),
        "y_train": y_train,
        "y_test": y_test,
    }


rng = np.random.default_rng(SEED)
n, d = 300, 3
W_TRUE = np.array([2.0, -1.0, 0.5])
B_TRUE = 1.0
X_syn = rng.standard_normal((n, d))
z_syn = X_syn @ W_TRUE + B_TRUE

y_syn_reg = z_syn + 0.5 * rng.standard_normal(n)
y_syn_clf = (rng.random(n) < 1 / (1 + np.exp(-z_syn))).astype(int)

X_dia, y_dia = load_diabetes(return_X_y=True)
X_bc, y_bc = load_breast_cancer(return_X_y=True)

REG_DATA = {
    "synthetic": split_and_scale(X_syn, y_syn_reg),
    "diabetes": split_and_scale(X_dia, y_dia),
}
CLF_DATA = {
    "synthetic": split_and_scale(X_syn, y_syn_clf, stratify=y_syn_clf),
    "breast_cancer": split_and_scale(X_bc, y_bc, stratify=y_bc),
}

for group in (REG_DATA, CLF_DATA):
    for name, s in group.items():
        assert s["X_train"].shape[0] == s["y_train"].shape[0]
        assert s["X_test"].shape[0] == s["y_test"].shape[0]
        assert np.allclose(s["X_train"].mean(axis=0), 0.0, atol=1e-10)
        assert np.allclose(s["X_train"].std(axis=0), 1.0)

for title, group in (("регрессия", REG_DATA), ("классификация", CLF_DATA)):
    print(title)
    for name, s in group.items():
        print(
            f"  {name:14s} train {s['X_train'].shape}, test {s['X_test'].shape}",
            f"| y_train mean {s['y_train'].mean():.4f}, y_test mean {s['y_test'].mean():.4f}",
        )
print("W_TRUE =", W_TRUE, " B_TRUE =", B_TRUE)
print("classes in synthetic y:", np.bincount(y_syn_clf))
print("synthetic X_train mean:", CLF_DATA["synthetic"]["X_train"].mean(axis=0))

## 2. Линейная регрессия

### 2.1. `solver="analytic"` без штрафа против `sklearn.linear_model.LinearRegression`

Обе модели решают одну задачу наименьших квадратов точным методом, поэтому
ожидается совпадение до ошибок округления (порядка `1e-13` и меньше), а не
просто «разумная близость». Сравниваются коэффициенты, `intercept`,
предсказания на test и метрики на test (MSE и R²), посчитанные одной и той же
функцией `sklearn.metrics` для обеих моделей.

In [ ]:
def max_abs_diff(a, b):
    """Наибольшее по модулю отличие двух массивов."""
    return float(np.max(np.abs(np.asarray(a) - np.asarray(b))))


def report_regression(title, own, sk, data):
    """Обучает обе модели на train и печатает сравнение на одних и тех же данных."""
    own.fit(data["X_train"], data["y_train"])
    sk.fit(data["X_train"], data["y_train"])
    pred_own = own.predict(data["X_test"])
    pred_sk = sk.predict(data["X_test"])
    y_test = data["y_test"]
    print(title)
    print(f"  макс |разность коэффициентов| : {max_abs_diff(own.coef_, sk.coef_):.2e}")
    print(f"  |разность intercept|          : {abs(own.intercept_ - sk.intercept_):.2e}")
    print(f"  макс |разность предсказаний|  : {max_abs_diff(pred_own, pred_sk):.2e}")
    print(f"  test MSE  свой / sklearn      : {mean_squared_error(y_test, pred_own):.6f} / {mean_squared_error(y_test, pred_sk):.6f}")
    print(f"  test R2   свой / sklearn      : {r2_score(y_test, pred_own):.6f} / {r2_score(y_test, pred_sk):.6f}")
    return own, sk

In [ ]:
for name, data in REG_DATA.items():
    own, sk = report_regression(
        f"{name}: analytic vs LinearRegression",
        LinearRegression(solver="analytic"),
        SkLinearRegression(),
        data,
    )
    assert max_abs_diff(own.coef_, sk.coef_) < 1e-8
    assert abs(own.intercept_ - sk.intercept_) < 1e-8
    if name == "synthetic":
        print("  свои coef_:", own.coef_, " intercept_:", round(own.intercept_, 6), " (истина", W_TRUE, B_TRUE, ")")

### 2.2. `solver="gradient"` без штрафа против `sklearn.linear_model.LinearRegression`

Параметры спуска: `learning_rate=0.4`, `max_iter=20000`, `tol=0.0`.

- `learning_rate=0.4` меньше границы устойчивости `2 / λ_max` (для `diabetes`
  она равна `0.510`, для синтетики `1.823`), где `λ_max` это наибольшее
  собственное число матрицы `XᵀX / n`.
- `tol=0.0` выключает раннюю остановку, чтобы сравнивать именно точность
  оптимизации, а не критерий остановки (его разберём в разделе про сходимость).
- `max_iter=20000` выбрано по замеру: на `diabetes` самое медленное направление
  затухает с множителем около `1 − 0.4 · 0.0094 ≈ 0.9962` за шаг, этого хватает
  для точности порядка `1e-12`.

Кроме сравнения с `sklearn` печатается значение цели `J` у `analytic` и
`gradient`: у точного решения и у сошедшегося спуска оно должно совпасть.

In [ ]:
GRAD_PARAMS = {"solver": "gradient", "learning_rate": 0.4, "max_iter": 20000, "tol": 0.0}

for name, data in REG_DATA.items():
    own, sk = report_regression(
        f"{name}: gradient vs LinearRegression",
        LinearRegression(**GRAD_PARAMS),
        SkLinearRegression(),
        data,
    )
    ana = LinearRegression(solver="analytic").fit(data["X_train"], data["y_train"])
    print(f"  n_iter_: {own.n_iter_}")
    print(f"  J (цель) analytic / gradient  : {ana.loss_history_[-1]:.6f} / {own.loss_history_[-1]:.6f}")
    assert own.n_iter_ == GRAD_PARAMS["max_iter"]
    assert max_abs_diff(own.coef_, sk.coef_) < 1e-8
    assert abs(own.intercept_ - sk.intercept_) < 1e-8

### 2.3. L2: `analytic` и `gradient` против `sklearn.linear_model.Ridge`

Цели записаны по-разному. Наша: `(1/2n) · Σ (zᵢ − yᵢ)² + alpha · ½ ‖w‖²`.
У `Ridge`: `Σ (zᵢ − yᵢ)² + α_sk · ‖w‖²`. Если умножить нашу цель на `2n`,
получится `Σ (zᵢ − yᵢ)² + n · alpha · ‖w‖²`, поэтому **`α_sk = n · alpha`**,
где `n` это число объектов в train. Свободный член не штрафуется ни у нас, ни у
`Ridge`.

В таблице: наибольшая по модулю разность коэффициентов, `intercept` и предсказаний
на test, затем MSE на test для нашей модели и `sklearn`. Последняя строка
показывает, что бывает, если забыть множитель `n`.

Для L2 граница устойчивости спуска сдвигается: `learning_rate < 2 / (λ_max + alpha)`.
Для `diabetes` при `alpha = 0.5` она равна `0.4526`, наш `learning_rate = 0.4` ниже.

In [ ]:
def regression_row(own, sk, data):
    """Обучает обе модели на train и возвращает числа для одной строки сравнения."""
    own.fit(data["X_train"], data["y_train"])
    sk.fit(data["X_train"], data["y_train"])
    pred_own = own.predict(data["X_test"])
    pred_sk = sk.predict(data["X_test"])
    y_test = data["y_test"]
    return {
        "coef": max_abs_diff(own.coef_, sk.coef_),
        "intercept": abs(own.intercept_ - sk.intercept_),
        "pred": max_abs_diff(pred_own, pred_sk),
        "mse_own": mean_squared_error(y_test, pred_own),
        "mse_sk": mean_squared_error(y_test, pred_sk),
    }

In [ ]:
ALPHAS = (0.1, 0.5)

print(f"{'набор':10s} {'alpha':>5s} {'solver':>9s} {'coef':>9s} {'intercept':>9s} {'pred':>9s}   test MSE свой / sklearn")
for name, data in REG_DATA.items():
    n_train = data["X_train"].shape[0]
    for alpha in ALPHAS:
        for solver_params in ({"solver": "analytic"}, GRAD_PARAMS):
            own = LinearRegression(penalty="l2", alpha=alpha, **solver_params)
            sk = SkRidge(alpha=n_train * alpha)
            row = regression_row(own, sk, data)
            print(
                f"{name:10s} {alpha:5.1f} {solver_params['solver']:>9s} "
                f"{row['coef']:9.2e} {row['intercept']:9.2e} {row['pred']:9.2e}"
                f"   {row['mse_own']:.6f} / {row['mse_sk']:.6f}"
            )
            assert row["coef"] < 1e-8 and row["intercept"] < 1e-8

data = REG_DATA["diabetes"]
wrong = regression_row(
    LinearRegression(penalty="l2", alpha=0.5, solver="analytic"), SkRidge(alpha=0.5), data
)
print(f"diabetes, alpha=0.5, Ridge(alpha=alpha) без множителя n: макс |разность коэффициентов| = {wrong['coef']:.3f}")

## 3. Логистическая регрессия

Сравнение с `sklearn.linear_model.LogisticRegression`. Параметр `penalty` у
`sklearn` устарел (`FutureWarning` с версии 1.8, удаление в 1.10), поэтому
режимы задаются через `C` и `l1_ratio`:

| Режим | Наша модель | `sklearn` |
|---|---|---|
| без штрафа | `penalty=None` | `C=np.inf` |
| L2 | `penalty="l2", alpha=alpha` | `C=1/(n·alpha), l1_ratio=0` |
| L1 | `penalty="l1", alpha=alpha` | `C=1/(n·alpha), l1_ratio=1, solver="saga"` |

Здесь `n` это число объектов в train. Для L1 выбран `saga`: `liblinear`
штрафует свободный член, а у нас и у `saga` он не штрафуется; `lbfgs` L1 не
поддерживает. Параметры нашего спуска: `learning_rate=0.5`, `max_iter=20000`,
`tol=0.0`.

Для проверки точности добавлен независимый эталон: точное решение методом
Ньютона (не использует ни наши классы, ни `sklearn`).

### 3.1. Без штрафа, синтетика

Сравниваются коэффициенты, `intercept`, `predict_proba`, `predict` и accuracy
на test. Отдельно считается расстояние каждой модели до точного решения.

In [ ]:
LOGIT_PARAMS = {"learning_rate": 0.5, "max_iter": 20000, "tol": 0.0}


def newton_reference(X, y, alpha=0.0, max_iter=100):
    """Точное решение (метод Ньютона) для логистической регрессии с L2-штрафом на w.

    Независимый эталон: не использует ни наши классы, ни sklearn.
    Возвращает (coef, intercept). Свободный член не штрафуется.
    """
    n, d = X.shape
    Xa = np.hstack([X, np.ones((n, 1))])
    theta = np.zeros(d + 1)
    R = np.diag([alpha] * d + [0.0])
    for _ in range(max_iter):
        p = 1 / (1 + np.exp(-(Xa @ theta)))
        grad = Xa.T @ (p - y) / n + R @ theta
        hess = (Xa.T * (p * (1 - p))) @ Xa / n + R
        step = np.linalg.solve(hess, grad)
        theta = theta - step
        if np.max(np.abs(step)) < 1e-15:
            break
    return theta[:d], theta[d]


def classification_row(own, sk, data):
    """Обучает обе модели на train и возвращает числа для одной строки сравнения."""
    own.fit(data["X_train"], data["y_train"])
    sk.fit(data["X_train"], data["y_train"])
    X_test, y_test = data["X_test"], data["y_test"]
    pred_own, pred_sk = own.predict(X_test), sk.predict(X_test)
    return {
        "coef": max_abs_diff(own.coef_, sk.coef_.ravel()),
        "intercept": abs(own.intercept_ - sk.intercept_[0]),
        "proba": max_abs_diff(own.predict_proba(X_test), sk.predict_proba(X_test)),
        "agree": np.mean(pred_own == pred_sk),
        "acc_own": np.mean(pred_own == y_test),
        "acc_sk": np.mean(pred_sk == y_test),
    }

In [ ]:
data = CLF_DATA["synthetic"]
own = LogisticRegression(**LOGIT_PARAMS)
sk = SkLogisticRegression(C=np.inf, tol=1e-12, max_iter=10000)
row = classification_row(own, sk, data)
w_ref, b_ref = newton_reference(data["X_train"], data["y_train"])
dist_own = max(max_abs_diff(own.coef_, w_ref), abs(own.intercept_ - b_ref))
dist_sk = max(max_abs_diff(sk.coef_.ravel(), w_ref), abs(sk.intercept_[0] - b_ref))

print("synthetic, без штрафа: свой спуск vs LogisticRegression(C=np.inf)")
print(f"  макс |разность коэффициентов|      : {row['coef']:.2e}")
print(f"  |разность intercept|               : {row['intercept']:.2e}")
print(f"  макс |разность predict_proba|      : {row['proba']:.2e}")
print(f"  доля совпавших классов (predict)   : {row['agree']:.4f}")
print(f"  accuracy на test свой / sklearn    : {row['acc_own']:.4f} / {row['acc_sk']:.4f}")
print(f"  расстояние до точного решения (Ньютон), свой / sklearn: {dist_own:.2e} / {dist_sk:.2e}")
print("  свои coef_:", own.coef_, " intercept_:", round(own.intercept_, 6))
print("  sklearn coef_:", sk.coef_.ravel(), " intercept_:", round(float(sk.intercept_[0]), 6))
print("  n_iter_ свой:", own.n_iter_, " sklearn:", sk.n_iter_)
assert row["coef"] < 1e-6 and row["intercept"] < 1e-6 and row["proba"] < 1e-6
assert row["agree"] == 1.0
assert dist_own < 1e-10

### 3.2. Без штрафа, `breast_cancer`: расхождение и его причина

Обучающая часть этого набора линейно разделима: после того как все объекты
классифицированы правильно, дальнейшее увеличение весов только уменьшает
лосс. У цели **нет конечного минимума**, поэтому результат зависит лишь от
того, когда остановился алгоритм. Это видно по таблицам ниже: норма весов
растёт и у нашего спуска (с числом итераций), и у `sklearn` (с ужесточением
`tol`), а точность на train доходит до `1.0`.

Поэтому коэффициенты здесь не сравниваются: их разность определяется
критерием остановки, а не качеством реализации. Сравниваются предсказания:
классы совпадают почти везде, точность на test близка. Вероятности
отличаются сильнее, потому что у `sklearn` веса огромные и вероятности
прижаты к 0 и 1.

Есть и побочный эффект: у нашего спуска точность на test **падает** с
ростом числа итераций (`0.9790` при 1000, `0.9441` при 100000). Это одно
разбиение и 143 объекта, поэтому это иллюстрация, а не доказательство.
Практический вывод: для разделимых данных нужна регуляризация (L2 и L1 ниже).

In [ ]:
data = CLF_DATA["breast_cancer"]
X_train, y_train = data["X_train"], data["y_train"]
X_test, y_test = data["X_test"], data["y_test"]

print("breast_cancer, без штрафа")
print("наш спуск (learning_rate=0.5, tol=0.0): рост весов с числом итераций")
print(f"  {'max_iter':>8s} {'||w||':>8s} {'J':>10s} {'train acc':>10s} {'test acc':>9s}")
own_norms, own_losses = [], []
for max_iter in (1000, 5000, 20000, 100000):
    model = LogisticRegression(learning_rate=0.5, max_iter=max_iter, tol=0.0).fit(X_train, y_train)
    own_norms.append(np.linalg.norm(model.coef_))
    own_losses.append(model.loss_history_[-1])
    print(
        f"  {max_iter:8d} {own_norms[-1]:8.2f} {own_losses[-1]:10.3e} "
        f"{np.mean(model.predict(X_train) == y_train):10.4f} {np.mean(model.predict(X_test) == y_test):9.4f}"
    )

print("sklearn (C=np.inf): рост весов с ужесточением tol")
print(f"  {'tol':>8s} {'n_iter':>6s} {'||w||':>9s} {'train acc':>10s} {'test acc':>9s}")
sk_norms = []
for tol in (1e-4, 1e-8, 1e-12):
    sk_model = SkLogisticRegression(C=np.inf, tol=tol, max_iter=100000).fit(X_train, y_train)
    sk_norms.append(np.linalg.norm(sk_model.coef_))
    print(
        f"  {tol:8.0e} {sk_model.n_iter_[0]:6d} {sk_norms[-1]:9.2f} "
        f"{np.mean(sk_model.predict(X_train) == y_train):10.4f} {np.mean(sk_model.predict(X_test) == y_test):9.4f}"
    )

own = LogisticRegression(**LOGIT_PARAMS)
sk = SkLogisticRegression(C=np.inf, tol=1e-12, max_iter=100000)
row = classification_row(own, sk, data)
n_disagree = int(np.sum(own.predict(X_test) != sk.predict(X_test)))
print("итог на test: свой (max_iter=20000) vs sklearn (tol=1e-12)")
print(f"  макс |разность коэффициентов|  : {row['coef']:.1f}  (сравнивать коэффициенты нельзя)")
print(f"  макс |разность predict_proba|  : {row['proba']:.4f}")
print(f"  доля совпавших классов         : {row['agree']:.4f}  (не совпали {n_disagree} из {len(y_test)})")
print(f"  accuracy на test свой / sklearn: {row['acc_own']:.4f} / {row['acc_sk']:.4f}")
assert all(a < b for a, b in zip(own_norms, own_norms[1:]))
assert all(a > b for a, b in zip(own_losses, own_losses[1:]))
assert all(a < b for a, b in zip(sk_norms, sk_norms[1:]))
assert np.mean(sk.predict(X_train) == y_train) == 1.0
assert row["agree"] > 0.97

### 3.3. L2: наш `penalty="l2"` против `LogisticRegression(C=..., l1_ratio=0)`

Цель `sklearn` при `l1_ratio=0`: `C · Σ ℓᵢ + ½‖w‖²`. Наша: `(1/n) · Σ ℓᵢ + alpha · ½‖w‖²`.
Если разделить цель `sklearn` на `C · n`, штраф превращается в
`(1/(C·n)) · ½‖w‖²`, поэтому **`alpha = 1/(C·n)`, то есть `C = 1/(n·alpha)`**,
где `n` это число объектов в train. Свободный член не штрафуется ни у нас,
ни у `lbfgs`.

В таблице: разности коэффициентов, `intercept` и `predict_proba` (наша модель
против `sklearn`), доля совпавших классов, accuracy на test, а также расстояние
каждой модели до точного решения (метод Ньютона с тем же штрафом). Для L2
граница устойчивости спуска ниже, чем без штрафа (примерно `0.59` для
`breast_cancer` при `alpha=0.1`), наш `learning_rate=0.5` под ней.

In [ ]:
ALPHAS_CLF = (0.01, 0.1)

print(f"{'набор':14s} {'alpha':>5s} {'C':>8s} {'coef':>9s} {'intercept':>9s} {'proba':>9s} {'agree':>6s}   accuracy свой / sklearn   до эталона свой / sklearn")
for name, data in CLF_DATA.items():
    n_train = data["X_train"].shape[0]
    for alpha in ALPHAS_CLF:
        C = 1 / (n_train * alpha)
        own = LogisticRegression(penalty="l2", alpha=alpha, **LOGIT_PARAMS)
        sk = SkLogisticRegression(C=C, l1_ratio=0, tol=1e-12, max_iter=10000)
        row = classification_row(own, sk, data)
        w_ref, b_ref = newton_reference(data["X_train"], data["y_train"], alpha=alpha)
        dist_own = max(max_abs_diff(own.coef_, w_ref), abs(own.intercept_ - b_ref))
        dist_sk = max(max_abs_diff(sk.coef_.ravel(), w_ref), abs(sk.intercept_[0] - b_ref))
        print(
            f"{name:14s} {alpha:5.2f} {C:8.4f} {row['coef']:9.2e} {row['intercept']:9.2e} "
            f"{row['proba']:9.2e} {row['agree']:6.3f}   {row['acc_own']:.4f} / {row['acc_sk']:.4f}"
            f"         {dist_own:.1e} / {dist_sk:.1e}"
        )
        assert row["coef"] < 1e-5 and row["intercept"] < 1e-5 and row["proba"] < 1e-5
        assert row["agree"] == 1.0
        assert dist_own < 1e-9

### 3.4. L1: наш `penalty="l1"` против `LogisticRegression(C=..., l1_ratio=1, solver="saga")`

Соответствие параметров то же, что и для L2: `C = 1/(n·alpha)` (штраф `‖w‖₁`
делится на `C·n`). Решатель `saga`, потому что `liblinear` штрафует свободный
член, а `lbfgs` L1 не поддерживает.

Столбцы таблицы:

- `dJ` : наша цель минус цель `sklearn` на train (цель с L1: средний лосс +
  `alpha·Σ|w|`); не отрицательна, потому что `saga` ближе к минимуму;
- `coef_nz` : наибольшая разность коэффициентов по признакам, **ненулевым** у `sklearn`;
- `w_zero` : наибольший `|w|` нашей модели по признакам, равным **нулю** у
  `sklearn` (у нас они не нули);
- `lr*alpha` : масштаб колебаний субградиентного шага;
- `nnz_sk` : число ненулевых весов у `sklearn` из общего числа признаков.

Вес `w_j = 0` оптимален, если `|g_j| ≤ alpha`. `saga` обнуляет такие веса
проксимальным шагом, поэтому нули у него точные. Наш субградиентный спуск с
постоянным шагом колеблется около нуля с размахом порядка `lr·alpha` и
точного нуля не даёт. Это известное ограничение метода (описано в README):
уменьшение `learning_rate` сжимает колебания, но замедляет спуск; убывающий шаг
или проксимальный метод решили бы вопрос, но выходят за рамки задания.

In [ ]:
def l1_objective(coef, intercept, X, y, alpha):
    """Цель с L1: mean(логлосс) + alpha * sum|w| (свободный член не штрафуется)."""
    z = X @ coef + intercept
    return np.mean(np.logaddexp(0, z) - y * z) + alpha * np.sum(np.abs(coef))


ALPHAS_L1 = (0.01, 0.05)
L1_RUNS = (
    {"learning_rate": 0.5, "max_iter": 20000, "tol": 0.0},
    {"learning_rate": 0.1, "max_iter": 50000, "tol": 0.0},
)

print(f"{'набор':14s} {'alpha':>5s} {'lr':>4s} {'dJ':>10s} {'coef_nz':>9s} {'w_zero':>7s} {'lr*alpha':>8s} {'nnz_sk':>6s} {'agree':>6s}   accuracy свой / sklearn")
for name, data in CLF_DATA.items():
    X_train, y_train = data["X_train"], data["y_train"]
    n_train, d = X_train.shape
    for alpha in ALPHAS_L1:
        for params in L1_RUNS:
            own = LogisticRegression(penalty="l1", alpha=alpha, **params)
            sk = SkLogisticRegression(C=1 / (n_train * alpha), l1_ratio=1, solver="saga", tol=1e-10, max_iter=200000)
            row = classification_row(own, sk, data)
            w_sk = sk.coef_.ravel()
            zero = w_sk == 0
            d_j = l1_objective(own.coef_, own.intercept_, X_train, y_train, alpha) - l1_objective(w_sk, sk.intercept_[0], X_train, y_train, alpha)
            coef_nz = max_abs_diff(own.coef_[~zero], w_sk[~zero])
            w_zero = np.max(np.abs(own.coef_[zero])) if zero.any() else 0.0
            step = params["learning_rate"] * alpha
            print(
                f"{name:14s} {alpha:5.2f} {params['learning_rate']:4.1f} {d_j:10.2e} {coef_nz:9.2e} {w_zero:7.4f} "
                f"{step:8.4f} {int((~zero).sum()):3d}/{d:<2d} {row['agree']:6.3f}   {row['acc_own']:.4f} / {row['acc_sk']:.4f}"
            )
            assert d_j > -1e-9
            assert w_zero < 3 * step
            assert row["agree"] > 0.97
            if name == "synthetic" and alpha == 0.01:
                assert coef_nz < 1e-6

## 4. Режимы: штраф и `fit_intercept`

Секция закрывает пункт задания «без регуляризации, L1, L2, с `intercept` и без него». Без штрафа и с L2 всё уже проверено в разделах 2 и 3, поэтому здесь два дополнения: L1 для регрессии (4.1) и сводная матрица «модель × штраф × `fit_intercept`» (4.2).

### 4.1 Линейная регрессия с L1 против `Lasso`

`Lasso` минимизирует `1/(2n)·‖y − Xw − b‖² + α·‖w‖₁`. Это в точности наша цель, поэтому `alpha_sklearn = alpha` (без множителя `n`, в отличие от `Ridge`, где `alpha_sklearn = n·alpha`).

Ожидание то же, что в 3.4. Если у `Lasso` все коэффициенты ненулевые, решения совпадают до ошибок округления. Если часть коэффициентов `Lasso` обнуляет точно, наш субградиентный спуск колеблется около нуля с размахом порядка `learning_rate·alpha` и до нуля не доходит.

Столбцы: `dJ` — разность цели (наша минус sklearn, должна быть ≥ 0), `coef` — максимальная разность коэффициентов, `w_zero` — наибольший наш коэффициент там, где у `Lasso` ноль, `lr*alpha` — размер шага штрафа, `nnz_sk` — число ненулевых коэффициентов у `Lasso`.

In [ ]:
def lasso_objective(coef, intercept, X, y, alpha):
    """Цель L1-регрессии: 1/(2n)·Σ(z−y)² + alpha·Σ|w| (свободный член не штрафуется)."""
    n = X.shape[0]
    return np.sum((X @ coef + intercept - y) ** 2) / (2 * n) + alpha * np.sum(np.abs(coef))


ALPHAS_LASSO = (0.01, 0.1)
L1_REG_RUNS = (
    {"learning_rate": 0.4, "max_iter": 20000, "tol": 0.0},
    {"learning_rate": 0.1, "max_iter": 50000, "tol": 0.0},
)

print(f"{'набор':10s} {'alpha':>5s} {'lr':>4s} {'dJ':>10s} {'coef':>9s} {'w_zero':>7s} {'lr*alpha':>8s} {'nnz_sk':>6s}   test MSE свой / sklearn")
for name, data in REG_DATA.items():
    X_train, y_train = data["X_train"], data["y_train"]
    d = X_train.shape[1]
    for alpha in ALPHAS_LASSO:
        for params in L1_REG_RUNS:
            own = LinearRegression(solver="gradient", penalty="l1", alpha=alpha, **params)
            sk = SkLasso(alpha=alpha, tol=1e-12, max_iter=1_000_000)
            row = regression_row(own, sk, data)
            zero = sk.coef_ == 0
            d_j = lasso_objective(own.coef_, own.intercept_, X_train, y_train, alpha) - lasso_objective(sk.coef_, sk.intercept_, X_train, y_train, alpha)
            w_zero = np.max(np.abs(own.coef_[zero])) if zero.any() else 0.0
            step = params["learning_rate"] * alpha
            print(
                f"{name:10s} {alpha:5.2f} {params['learning_rate']:4.1f} {d_j:10.2e} {row['coef']:9.2e} {w_zero:7.4f} "
                f"{step:8.4f} {int((~zero).sum()):3d}/{d:<2d}   {row['mse_own']:.6f} / {row['mse_sk']:.6f}"
            )
            assert d_j > -1e-9
            assert w_zero < 3 * step
            if not zero.any():
                assert row["coef"] < 1e-6 and row["intercept"] < 1e-8

### 4.2 Сводная матрица: штраф × `fit_intercept`

Для каждого сочетания «штраф × solver × `fit_intercept`» сравниваем свою модель с sklearn при одном и том же `alpha = 0.01`. Пересчёт параметров такой, как в разделах выше:

| Задача | Штраф | Параметр sklearn |
|---|---|---|
| регрессия | L2 | `Ridge(alpha = n·alpha)` |
| регрессия | L1 | `Lasso(alpha = alpha)` |
| логистика | L2 / L1 | `C = 1/(n·alpha)` |

При `fit_intercept=False` свободный член равен нулю у обеих моделей: `analytic` не центрирует данные, `gradient` не обновляет `b`. Проверка `own.intercept_ == 0.0` в ячейке это подтверждает.

Ожидание: во всех строках разность коэффициентов не больше 1e-6 (на практике 1e-8 и меньше), у логистики полностью совпадают предсказанные классы.

In [ ]:
ALPHA_MATRIX = 0.01
REG_MODES = (
    # (подпись, kwargs своей модели, функция n_train -> sklearn-модель при fit_intercept)
    ("без штрафа, analytic", {"solver": "analytic"}, lambda n, fi: SkLinearRegression(fit_intercept=fi)),
    ("без штрафа, gradient", GRAD_PARAMS, lambda n, fi: SkLinearRegression(fit_intercept=fi)),
    ("L2, analytic", {"solver": "analytic", "penalty": "l2", "alpha": ALPHA_MATRIX},
     lambda n, fi: SkRidge(alpha=n * ALPHA_MATRIX, fit_intercept=fi)),
    ("L2, gradient", {**GRAD_PARAMS, "penalty": "l2", "alpha": ALPHA_MATRIX},
     lambda n, fi: SkRidge(alpha=n * ALPHA_MATRIX, fit_intercept=fi)),
    ("L1, gradient", {**GRAD_PARAMS, "penalty": "l1", "alpha": ALPHA_MATRIX},
     lambda n, fi: SkLasso(alpha=ALPHA_MATRIX, fit_intercept=fi, tol=1e-12, max_iter=1_000_000)),
)
CLF_MODES = (
    ("без штрафа", {},
     lambda n, fi: SkLogisticRegression(C=np.inf, fit_intercept=fi, tol=1e-12, max_iter=10000)),
    ("L2", {"penalty": "l2", "alpha": ALPHA_MATRIX},
     lambda n, fi: SkLogisticRegression(C=1 / (n * ALPHA_MATRIX), l1_ratio=0, fit_intercept=fi, tol=1e-12, max_iter=10000)),
    ("L1", {"penalty": "l1", "alpha": ALPHA_MATRIX},
     lambda n, fi: SkLogisticRegression(C=1 / (n * ALPHA_MATRIX), l1_ratio=1, solver="saga", random_state=SEED, fit_intercept=fi, tol=1e-10, max_iter=200000)),
)

data = REG_DATA["diabetes"]
n_train = data["X_train"].shape[0]
print(f"регрессия, diabetes, alpha={ALPHA_MATRIX}")
print(f"{'режим':22s} {'fit_int':>7s} {'coef':>9s} {'intercept':>9s} {'pred':>9s}   own.intercept_")
for label, params, make_sk in REG_MODES:
    for fit_intercept in (True, False):
        own = LinearRegression(fit_intercept=fit_intercept, **params)
        row = regression_row(own, make_sk(n_train, fit_intercept), data)
        print(
            f"{label:22s} {str(fit_intercept):>7s} {row['coef']:9.2e} {row['intercept']:9.2e} {row['pred']:9.2e}"
            f"   {own.intercept_:.4f}"
        )
        assert row["coef"] < 1e-6 and row["intercept"] < 1e-8
        if not fit_intercept:
            assert own.intercept_ == 0.0

data = CLF_DATA["synthetic"]
n_train = data["X_train"].shape[0]
print(f"\nклассификация, synthetic, alpha={ALPHA_MATRIX}")
print(f"{'режим':22s} {'fit_int':>7s} {'coef':>9s} {'intercept':>9s} {'proba':>9s} {'agree':>6s}   own.intercept_")
for label, params, make_sk in CLF_MODES:
    for fit_intercept in (True, False):
        own = LogisticRegression(fit_intercept=fit_intercept, **params, **LOGIT_PARAMS)
        row = classification_row(own, make_sk(n_train, fit_intercept), data)
        print(
            f"{label:22s} {str(fit_intercept):>7s} {row['coef']:9.2e} {row['intercept']:9.2e} {row['proba']:9.2e} {row['agree']:6.3f}"
            f"   {own.intercept_:.4f}"
        )
        assert row["coef"] < 1e-6 and row["intercept"] < 1e-6 and row["agree"] == 1.0
        if not fit_intercept:
            assert own.intercept_ == 0.0

## 5. Сходимость градиентного спуска

### 5.1 Регрессия: кривые `loss_history_` и влияние `learning_rate`

Для линейной регрессии без штрафа цель `J` квадратична, её гессиан равен `XaᵀXa/n` (`Xa` — это `X` со столбцом единиц для `intercept`). Шаг спуска умножает ошибку по каждому собственному направлению с собственным значением `λ` на `(1 − lr·λ)`. Отсюда два вывода:

- спуск сходится, только если `|1 − lr·λ| < 1` для всех `λ`, то есть `lr < 2/λmax`;
- скорость определяет самое маленькое `λmin`: на больших `k` отношение соседних `J − J*` равно `(1 − lr·λmin)²`. Чем хуже обусловлена задача (`λmax/λmin` велико), тем медленнее спуск.

`loss_history_` пишется после каждого шага, поэтому `J(0)` в историю не входит. Слева: сходимость при трёх `lr` в логарифмической шкале; тонкие чёрные линии — предсказание формулы. Справа: поведение вокруг границы `2/λmax`.

In [ ]:
data = REG_DATA["diabetes"]
X_train, y_train = data["X_train"], data["y_train"]
n_train = X_train.shape[0]

# Гессиан цели без штрафа: (1/n)·XaᵀXa, где Xa = [X, 1] (столбец единиц отвечает за intercept)
X_aug = np.hstack([X_train, np.ones((n_train, 1))])
eig = np.linalg.eigvalsh(X_aug.T @ X_aug / n_train)
lam_min, lam_max = eig[0], eig[-1]
lr_limit = 2 / lam_max
j_star = LinearRegression(solver="analytic").fit(X_train, y_train).loss_history_[-1]
print(f"diabetes: lambda_min = {lam_min:.5f}, lambda_max = {lam_max:.5f}, обусловленность = {lam_max / lam_min:.0f}")
print(f"граница устойчивости 2/lambda_max = {lr_limit:.4f};  J* (analytic) = {j_star:.6f}")

LR_LIST = (0.05, 0.2, 0.4)
STEPS = 3000
K = 1000  # где сверяем наблюдаемую скорость с формулой
history = {}
print(f"\n{'lr':>5s} {'J-J* (k=100)':>13s} {'J-J* (k=1000)':>14s} {'J-J* (k=3000)':>14s} {'наблюд. отношение':>18s} {'формула':>10s}")
for lr in LR_LIST:
    model = LinearRegression(solver="gradient", learning_rate=lr, max_iter=STEPS, tol=0.0).fit(X_train, y_train)
    excess = np.array(model.loss_history_) - j_star
    history[lr] = excess
    observed = excess[K + 1] / excess[K]
    predicted = (1 - lr * lam_min) ** 2
    print(f"{lr:5.2f} {excess[99]:13.4e} {excess[K - 1]:14.4e} {excess[STEPS - 1]:14.4e} {observed:18.8f} {predicted:10.8f}")
    assert abs(observed - predicted) < 1e-7

# Расходимость: lr около границы 2/lambda_max и выше неё
DIV_LR = (0.40, 0.50, 0.52, 0.55)
DIV_STEPS = 200
div_history = {}
print(f"\n{'lr':>5s} {'lr*lambda_max':>13s} {'|1-lr*lambda_max|':>18s} {'J (k=1)':>10s} {'J (k=200)':>12s}")
for lr in DIV_LR:
    model = LinearRegression(solver="gradient", learning_rate=lr, max_iter=DIV_STEPS, tol=0.0).fit(X_train, y_train)
    div_history[lr] = np.array(model.loss_history_)
    h = div_history[lr]
    print(f"{lr:5.2f} {lr * lam_max:13.4f} {abs(1 - lr * lam_max):18.4f} {h[0]:10.1f} {h[-1]:12.4e}")
assert div_history[0.40][-1] < div_history[0.40][0] and div_history[0.50][-1] < div_history[0.50][0]
assert div_history[0.52][-1] > 100 * div_history[0.52][0] and div_history[0.55][-1] > 100 * div_history[0.55][0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
colors = {0.05: "#2a78d6", 0.2: "#eb6834", 0.4: "#1baf7a"}
styles = {0.05: "-", 0.2: "--", 0.4: ":"}
for lr in LR_LIST:
    ax1.semilogy(np.arange(1, STEPS + 1), history[lr], color=colors[lr], ls=styles[lr], lw=2, label=f"lr = {lr}")
    k = np.arange(K, STEPS + 1)
    ax1.semilogy(k + 1, history[lr][K] * (1 - lr * lam_min) ** (2 * (k - K)), color="black", lw=0.8, alpha=0.6)
ax1.plot([], [], color="black", lw=0.8, alpha=0.6, label="формула (1 − lr·λmin)²ᵏ")
ax1.set_xlabel("итерация k")
ax1.set_ylabel("J − J* (log)")
ax1.set_title("diabetes: сходимость при разных lr")
ax1.legend(frameon=False)
ax1.grid(alpha=0.25)

dcol = {0.40: "#2a78d6", 0.50: "#1baf7a", 0.52: "#eb6834", 0.55: "#7a3b8f"}
for lr in DIV_LR:
    ax2.semilogy(np.arange(1, DIV_STEPS + 1), div_history[lr], color=dcol[lr], lw=2, ls="-" if lr < lr_limit else "--", label=f"lr = {lr:.2f}")
ax2.set_xlabel("итерация k")
ax2.set_ylabel("J (log)")
ax2.set_title(f"граница устойчивости 2/λmax = {lr_limit:.4f}")
ax2.legend(frameon=False)
ax2.grid(alpha=0.25)
for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()